# 📝 Éditeur Interactif de Coordonnées

Ce notebook vous permet de modifier facilement les coordonnées dans le fichier `submissions_reordered.csv`.

## 🎯 Fonctionnalités
- Visualiser toutes les coordonnées
- Détecter les coordonnées non conformes
- Modifier des lignes spécifiques
- Valider les modifications
- Sauvegarder le fichier corrigé

In [ ]:
# Import des bibliothèques nécessaires
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Bibliothèques importées avec succès")

## 📁 Chargement des données

In [ ]:
# Charger le fichier CSV
csv_file = 'submissions_TheThinkers.csv'

try:
    # Lire le CSV avec pandas
    df = pd.read_csv(csv_file)
    print(f"✅ Fichier chargé: {len(df)} lignes")
    print(f"📋 Colonnes: {list(df.columns)}")
    
    # Afficher les premières lignes
    display(df.head())
    
except FileNotFoundError:
    print("❌ Fichier submissions_TheThinkers.csv introuvable")
    print("💡 Assurez-vous d'être dans le bon dossier")
except Exception as e:
    print(f"❌ Erreur lors du chargement: {e}")

## 🔍 Fonctions utilitaires

In [ ]:
def parse_coordinates(coord_string: str) -> List[Dict]:
    """Parse une chaîne de coordonnées JSON"""
    if pd.isna(coord_string) or coord_string == '':
        return []
    
    try:
        # Nettoyer la chaîne (remplacer les doubles guillemets)
        cleaned = coord_string.replace('""', '"')
        coords = json.loads(cleaned)
        return coords if isinstance(coords, list) else []
    except:
        return []

def validate_coordinate(x: float, y: float) -> Tuple[bool, str]:
    """Valide une coordonnée UTM"""
    issues = []
    
    # Vérifier les plages UTM pour le Bénin (Zone 31N)
    if not (350000 <= x <= 550000):
        issues.append(f"X={x} hors plage normale (350000-550000)")
    
    if not (650000 <= y <= 850000):
        issues.append(f"Y={y} hors plage normale (650000-850000)")
    
    # Vérifier les valeurs suspectes
    if x == 0 or y == 0:
        issues.append("Coordonnée nulle détectée")
    
    if len(str(int(x))) < 6 or len(str(int(y))) < 6:
        issues.append("Coordonnée trop courte")
    
    return len(issues) == 0, "; ".join(issues)

def analyze_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    """Analyse toutes les coordonnées du DataFrame"""
    analysis = []
    
    for idx, row in df.iterrows():
        coords = parse_coordinates(row['Coordonnées'])
        
        if not coords:
            analysis.append({
                'Index': idx,
                'Nom_du_levé': row['Nom_du_levé'],
                'Nb_coordonnées': 0,
                'Coordonnées_valides': False,
                'Problèmes': 'Aucune coordonnée trouvée'
            })
            continue
        
        all_valid = True
        all_issues = []
        
        for i, coord in enumerate(coords):
            x = coord.get('x', 0)
            y = coord.get('y', 0)
            valid, issues = validate_coordinate(x, y)
            
            if not valid:
                all_valid = False
                all_issues.append(f"Coord {i+1}: {issues}")
        
        analysis.append({
            'Index': idx,
            'Nom_du_levé': row['Nom_du_levé'],
            'Nb_coordonnées': len(coords),
            'Coordonnées_valides': all_valid,
            'Problèmes': "; ".join(all_issues) if all_issues else 'OK'
        })
    
    return pd.DataFrame(analysis)

print("✅ Fonctions utilitaires définies")

## 📊 Analyse des coordonnées existantes

In [ ]:
# Analyser toutes les coordonnées
if 'df' in locals():
    print("🔍 Analyse des coordonnées en cours...")
    analysis_df = analyze_coordinates(df)
    
    # Statistiques générales
    print(f"📊 STATISTIQUES GÉNÉRALES")
    print(f"Total d'images: {len(analysis_df)}")
    print(f"Images avec coordonnées valides: {sum(analysis_df['Coordonnées_valides'])}")
    print(f"Images avec problèmes: {sum(~analysis_df['Coordonnées_valides'])}")
    
    # Afficher les problèmes détectés
    problemes = analysis_df[~analysis_df['Coordonnées_valides']]
    
    if len(problemes) > 0:
        print(f"\n❌ {len(problemes)} IMAGES AVEC PROBLÈMES:")
        display(problemes[['Index', 'Nom_du_levé', 'Nb_coordonnées', 'Problèmes']])
    else:
        print("\n✅ Toutes les coordonnées sont valides !")
        
    # Sauvegarde de l'analyse
    analysis_df.to_csv('analyse_coordonnees.csv', index=False)
    print("\n💾 Analyse sauvegardée dans 'analyse_coordonnees.csv'")
else:
    print("❌ Données non chargées")

## 🎯 Visualisation des coordonnées

In [ ]:
# Extraire toutes les coordonnées pour visualisation
def extract_all_coordinates(df: pd.DataFrame) -> Tuple[List[float], List[float], List[str]]:
    """Extrait toutes les coordonnées pour la visualisation"""
    x_coords, y_coords, labels = [], [], []
    
    for idx, row in df.iterrows():
        coords = parse_coordinates(row['Coordonnées'])
        for coord in coords:
            x_coords.append(coord.get('x', 0))
            y_coords.append(coord.get('y', 0))
            labels.append(row['Nom_du_levé'])
    
    return x_coords, y_coords, labels

if 'df' in locals():
    x_coords, y_coords, labels = extract_all_coordinates(df)
    
    if x_coords and y_coords:
        # Créer la visualisation
        plt.figure(figsize=(12, 8))
        
        # Scatter plot des coordonnées
        scatter = plt.scatter(x_coords, y_coords, alpha=0.6, c=range(len(x_coords)), cmap='viridis')
        plt.colorbar(scatter, label='Index des coordonnées')
        
        plt.xlabel('X (UTM)')
        plt.ylabel('Y (UTM)')
        plt.title('Distribution des Coordonnées UTM')
        plt.grid(True, alpha=0.3)
        
        # Ajouter les limites de validité
        plt.axvline(x=350000, color='red', linestyle='--', alpha=0.5, label='Limites X')
        plt.axvline(x=550000, color='red', linestyle='--', alpha=0.5)
        plt.axhline(y=650000, color='red', linestyle='--', alpha=0.5, label='Limites Y')
        plt.axhline(y=850000, color='red', linestyle='--', alpha=0.5)
        
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        print(f"📍 {len(x_coords)} coordonnées visualisées")
        print(f"🔍 Plage X: {min(x_coords):.0f} à {max(x_coords):.0f}")
        print(f"🔍 Plage Y: {min(y_coords):.0f} à {max(y_coords):.0f}")
    else:
        print("❌ Aucune coordonnée à visualiser")
else:
    print("❌ Données non chargées")

## ✏️ Interface de modification

In [ ]:
def show_row_details(df: pd.DataFrame, index: int):
    """Affiche les détails d'une ligne spécifique"""
    if index >= len(df) or index < 0:
        print(f"❌ Index {index} invalide (max: {len(df)-1})")
        return None
    
    row = df.iloc[index]
    coords = parse_coordinates(row['Coordonnées'])
    
    print(f"📋 LIGNE {index}")
    print(f"Nom du levé: {row['Nom_du_levé']}")
    print(f"Nombre de coordonnées: {len(coords)}")
    
    if coords:
        print("Coordonnées actuelles:")
        for i, coord in enumerate(coords):
            x, y = coord.get('x', 0), coord.get('y', 0)
            valid, issues = validate_coordinate(x, y)
            status = "✅" if valid else "❌"
            print(f"  {i+1}. X={x}, Y={y} {status}")
            if not valid:
                print(f"     Problème: {issues}")
    else:
        print("❌ Aucune coordonnée trouvée")
    
    return row

def update_coordinates(df: pd.DataFrame, index: int, new_coordinates: List[Dict]) -> pd.DataFrame:
    """Met à jour les coordonnées d'une ligne"""
    if index >= len(df) or index < 0:
        print(f"❌ Index {index} invalide")
        return df
    
    # Valider les nouvelles coordonnées
    valid_coords = []
    for coord in new_coordinates:
        if 'x' in coord and 'y' in coord:
            x, y = float(coord['x']), float(coord['y'])
            valid, issues = validate_coordinate(x, y)
            if valid:
                valid_coords.append({'x': x, 'y': y})
            else:
                print(f"⚠️ Coordonnée ignorée X={x}, Y={y}: {issues}")
    
    if valid_coords:
        # Convertir en string JSON avec le bon format
        coord_string = json.dumps(valid_coords).replace('"', '""')
        df.loc[index, 'Coordonnées'] = coord_string
        print(f"✅ {len(valid_coords)} coordonnées mises à jour pour la ligne {index}")
    else:
        print("❌ Aucune coordonnée valide à mettre à jour")
    
    return df

print("✅ Fonctions de modification définies")

## 🔧 Zone de modification interactive

Utilisez les cellules ci-dessous pour modifier les coordonnées problématiques.

In [ ]:
# ÉTAPE 1: Voir les détails d'une ligne spécifique
# Changez l'index pour voir une autre ligne
ligne_a_modifier = 0  # ⬅️ Changez cet index

if 'df' in locals():
    show_row_details(df, ligne_a_modifier)
else:
    print("❌ Données non chargées")

In [ ]:
# ÉTAPE 2: Modifier les coordonnées
# Exemple de modification pour la ligne sélectionnée ci-dessus

# Nouvelles coordonnées à appliquer (format: liste de dictionnaires)
nouvelles_coordonnees = [
    {"x": 427094.7, "y": 712773.67},
    {"x": 427110.61, "y": 712767.66},
    {"x": 427103.58, "y": 712748.94},
    {"x": 427099.06, "y": 712746.69},
    {"x": 427084.21, "y": 712750.65}
]

# ⚠️ DÉCOMMENTEZ LA LIGNE CI-DESSOUS POUR APPLIQUER LES MODIFICATIONS
# df = update_coordinates(df, ligne_a_modifier, nouvelles_coordonnees)

print("💡 Décommentez la ligne ci-dessus pour appliquer les modifications")

In [ ]:
# ÉTAPE 3: Vérifier les modifications
if 'df' in locals():
    print("🔍 Vérification après modification:")
    show_row_details(df, ligne_a_modifier)
else:
    print("❌ Données non chargées")

## 🔍 Exemples de modifications courantes

In [ ]:
# EXEMPLE 1: Corriger des coordonnées avec des valeurs aberrantes
def fix_common_issues(df: pd.DataFrame) -> pd.DataFrame:
    """Corrige automatiquement les problèmes courants"""
    df_fixed = df.copy()
    fixes_applied = 0
    
    for idx, row in df_fixed.iterrows():
        coords = parse_coordinates(row['Coordonnées'])
        if not coords:
            continue
            
        modified = False
        fixed_coords = []
        
        for coord in coords:
            x, y = coord.get('x', 0), coord.get('y', 0)
            
            # Correction 1: Valeurs nulles suspectes
            if x == 0 or y == 0:
                print(f"⚠️ Ligne {idx}: Coordonnée nulle ignorée X={x}, Y={y}")
                continue
            
            # Correction 2: Coordonnées trop courtes (ajouter des zéros)
            if len(str(int(x))) < 6 and x > 0:
                x = x * 10 ** (6 - len(str(int(x))))
                modified = True
                print(f"🔧 Ligne {idx}: X corrigé {coord.get('x')} → {x}")
            
            if len(str(int(y))) < 6 and y > 0:
                y = y * 10 ** (6 - len(str(int(y))))
                modified = True
                print(f"🔧 Ligne {idx}: Y corrigé {coord.get('y')} → {y}")
            
            # Garder les coordonnées valides
            valid, _ = validate_coordinate(x, y)
            if valid:
                fixed_coords.append({'x': x, 'y': y})
        
        # Mettre à jour si des modifications ont été faites
        if modified and fixed_coords:
            coord_string = json.dumps(fixed_coords).replace('"', '""')
            df_fixed.loc[idx, 'Coordonnées'] = coord_string
            fixes_applied += 1
    
    print(f"✅ {fixes_applied} lignes corrigées automatiquement")
    return df_fixed

# ⚠️ DÉCOMMENTEZ POUR APPLIQUER LES CORRECTIONS AUTOMATIQUES
# if 'df' in locals():
#     df = fix_common_issues(df)

print("💡 Fonction de correction automatique définie")

## 💾 Sauvegarde des modifications

In [ ]:
# Sauvegarder les modifications
def save_corrected_csv(df: pd.DataFrame, filename: str = 'submissions_reordered_corrected.csv'):
    """Sauvegarde le DataFrame corrigé"""
    try:
        # Faire une sauvegarde de l'original
        import shutil
        shutil.copy2('submissions_TheThinkers.csv', 'submissions_reordered_backup.csv')
        print("✅ Sauvegarde de l'original créée: submissions_reordered_backup.csv")
        
        # Sauvegarder la version corrigée
        df.to_csv(filename, index=False)
        print(f"✅ Version corrigée sauvegardée: {filename}")
        
        # Analyser la version corrigée
        print("\n🔍 Analyse de la version corrigée:")
        corrected_analysis = analyze_coordinates(df)
        valid_count = sum(corrected_analysis['Coordonnées_valides'])
        total_count = len(corrected_analysis)
        
        print(f"📊 Résultats: {valid_count}/{total_count} images avec coordonnées valides")
        
        remaining_issues = corrected_analysis[~corrected_analysis['Coordonnées_valides']]
        if len(remaining_issues) > 0:
            print(f"⚠️ {len(remaining_issues)} images nécessitent encore des corrections:")
            display(remaining_issues[['Index', 'Nom_du_levé', 'Problèmes']])
        else:
            print("🎉 Toutes les coordonnées sont maintenant valides !")
        
        return True
        
    except Exception as e:
        print(f"❌ Erreur lors de la sauvegarde: {e}")
        return False

# ⚠️ DÉCOMMENTEZ POUR SAUVEGARDER VOS MODIFICATIONS
# if 'df' in locals():
#     save_corrected_csv(df)

print("💡 Fonction de sauvegarde définie")
print("⚠️ N'oubliez pas de décommenter la ligne ci-dessus pour sauvegarder !")

## 🎯 Guide d'utilisation rapide

### Pour modifier une ligne spécifique:

1. **Identifier la ligne**: Regardez l'analyse ci-dessus pour voir les lignes avec problèmes
2. **Voir les détails**: Changez `ligne_a_modifier` dans la cellule correspondante
3. **Préparer les nouvelles coordonnées**: Modifiez la liste `nouvelles_coordonnees`
4. **Appliquer**: Décommentez la ligne `df = update_coordinates(...)`
5. **Vérifier**: Exécutez la cellule de vérification
6. **Sauvegarder**: Décommentez `save_corrected_csv(df)` quand vous avez fini

### Format des coordonnées:
```python
nouvelles_coordonnees = [
    {"x": 427094.7, "y": 712773.67},   # Première coordonnée
    {"x": 427110.61, "y": 712767.66}   # Deuxième coordonnée
    # ... ajoutez autant que nécessaire
]
```

### Plages valides (UTM Zone 31N - Bénin):
- **X**: 350,000 à 550,000
- **Y**: 650,000 à 850,000